# Save the Human (in the loop): when does human oversight of AI actually reduce risk? — Results

This notebook shows the results of the study set out in the project outline. It computes them itself, section by section, from the public FiFAR dataset. Starting from the dataset's alerts file and the framework's Classifier h, it regenerates the simulated reviewers, draws the teams, runs the three policies, re-draws the cases to give every difference a range, and applies the rules that decide the hypothesis. The outputs below are those of the run on free Google Colab that produced this copy. The notebook puts the deciding results first, then the descriptive ones; the problem and the objective are set out in the report. Where the study departs from the project outline, the departure is explained where it first matters. Technical records (run time and memory of this run, input checksums, and how each item traces to the study's design) are at the end and are not needed to read the results.

**Data.** Financial Fraud Alert Review Dataset (FiFAR), version 1 (Alves et al., 2025), figshare, digital object identifier 10.6084/m9.figshare.28351172 (https://doi.org/10.6084/m9.figshare.28351172), licensed under CC BY 4.0 (https://creativecommons.org/licenses/by/4.0/). The dataset is described in Alves et al., "A benchmarking framework and dataset for learning to defer in human-AI decision-making", *Scientific Data* 12, 506 (2025), https://doi.org/10.1038/s41597-025-04664-y. The notebook downloads the dataset and uses it unmodified; the repository does not redistribute it. The dataset's `Base.csv` derives from the Bank Account Fraud dataset, which is credited in the README, section "Data: FiFAR" ([https://github.com/zsipka/SaveHITL-pub#data-fifar](https://github.com/zsipka/SaveHITL-pub#data-fifar)); this notebook downloads that file and checks its checksum. The results are computed from FiFAR by this project and are not part of it. Of `Base.csv` itself, the notebook shows only aggregate figures (rates, counts and correlations over groups of at least 10 applications); no application row or individual value is shown.

The tables and graphs this notebook produces are computed from FiFAR (Alves et al., figshare, 2025, https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR by this project, not part of it, and FiFAR's creators have not reviewed or endorsed them.

**Files in `results/`: derived from FiFAR.** This project made the three files in `results/` from its runs on FiFAR (https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR, not part of it.

- `router_selected_settings.csv`: the settings chosen for each of the 250 router models, with the weighted log-loss of the selected trial (column `selected_weighted_log_loss`), which this project computed on FiFAR.
- `router_model_checksums.csv`: the checksums of the 250 router models this project trained on FiFAR.
- `input_checksums.csv`: the paths, sizes and checksums of the input files, 132 from FiFAR and 10 from OpenL2D (Apache License 2.0, see `NOTICE`).

They hold no FiFAR case row and no data value from FiFAR: no feature, label, score, reviewer decision or reviewer parameter. To show what each row refers to, they reuse FiFAR's own reviewer labels (for example `standard#13`) and FiFAR's file paths and file sizes. FiFAR's creators have not reviewed or endorsed these files.

## Contents

- [1. Running the notebook](#1-running-the-notebook)
- [2. The data and the set-up](#2-the-data-and-the-set-up)
- [3. Hypothesis](#3-hypothesis)
- [4. Descriptive results](#4-descriptive-results)
- [5. Limitations and risks](#5-limitations-and-risks)
- [The switch in this run](#the-switch-in-this-run)
- [Input checksums](#input-checksums)
- [Map from each item to its design identifier](#map-from-each-item-to-its-design-identifier)
- [Label texts](#label-texts)
- [The results that decide, in full](#the-results-that-decide-in-full)
- [Boundary summaries: caveat](#boundary-summaries-caveat)
- [Run time and memory of this run](#run-time-and-memory-of-this-run)
- [This run's outputs in one archive](#this-runs-outputs-in-one-archive)

<a id="1-running-the-notebook"></a>
## 1. Running the notebook

**What the run does.** Open the notebook in Google Colab from [the repository](https://github.com/zsipka/SaveHITL-pub), or run it on your own machine with the repository's environment file; Colab is the tested route. Run the cells in order. The run:
1. sets up the environment;
2. clones the framework and applies the project's four patches;
3. downloads the FiFAR data from figshare;
4. computes every section in turn.

We chose not to retrain all models from the framework for efficiency reasons. The models used and trained are introduced in [The data and the set-up](#2-the-data-and-the-set-up). We have pre-run the hyper parameter search for the trained models. The Notebook is loading the settings from this search by default but leaves the option to re-run the search (see the switch below).

**Permission to download.** The next cell asks for your permission to download FiFAR from figshare. Nothing is downloaded until you give it.

In [ ]:
# Settings
FIFAR_DOWNLOAD_ALLOWED = False   # set to True to give permission to download FiFAR from figshare (CC BY 4.0);
                                 # nothing is downloaded, and the run stops at the set-up, until it is True

**The switch**:

In [ ]:
# HYPERPARAM_TUNE
# False (default): the router's 250 models are refitted by the framework's own
#   procedure from the settings the study's settings search chose
#   (results/router_selected_settings.csv). This reproduces the study's figures.
# True: the framework's full settings search runs first, 120 trials per model,
#   then the run continues as normal. About 7 hours on a typical laptop;
#   14 to 21 hours on free Colab, which is more than one free session.
HYPERPARAM_TUNE = False

In [ ]:
# The repository: this notebook's own folder when it is run from a clone of SaveHITL-pub; otherwise (Colab) a fresh
# clone. Then the run's layout and the state of the switch.
import os, sys, subprocess
REPOSITORY = "https://github.com/zsipka/SaveHITL-pub.git"
if not os.path.isdir("savehitl"):
    if not os.path.isdir("SaveHITL-pub"):
        subprocess.run(["git", "clone", "-q", REPOSITORY, "SaveHITL-pub"], check=True)
    os.chdir("SaveHITL-pub")
sys.path.insert(0, os.getcwd())
from savehitl import notebook as nb
nb.start(HYPERPARAM_TUNE);

In [ ]:
# Environment (Python 3.7.9 and the pinned versions), the framework at its pinned commit with the four patches, and
# FiFAR from figshare; every file is stopped on its sha256. Each part runs once per session.
nb.setup(FIFAR_DOWNLOAD_ALLOWED)

<a id="2-the-data-and-the-set-up"></a>
## 2. The data and the set-up

**The data.** The study uses FiFAR, which is built on the Bank Account Fraud dataset: 1,000,000 generated bank account applications over eight months, each labelled as fraud or legitimate (file `Base.csv`). The dataset's Alert Model scores every application from month 3 on and flags those above a cut-off, set so that 5% of the legitimate applications of month 3 are flagged. The 30,622 flagged applications are the alerts, and only alerts reach Classifier h or a reviewer. The results are measured on month 7, the test month, whose 4,457 alerts (711 frauds, 3,746 legitimate) are the cases the policies decide.

Frauds that the alert model does not flag are ignored in this study. Frauds can be controlled by the alert threshold in practice. We use fixed threshold at 5% false positive rate.

**Which months are used for what.** Months are numbered 0 to 7, as in the data files. The graph shows which months each model and the reviewers learn from or are set on, and which month gives the results. Month 7, the test month, is used only for results: nothing is learned, chosen or set on it.

> **Runtime estimate (Colab free): not yet measured**
>
> The next cell computes the aggregate figures of `Base.csv` (1,000,000 applications), which both graphs of this section read.

In [ ]:
nb.section("F-months")

These figures summarise FiFAR's `Base.csv` in aggregate: rates, counts and correlations over groups of at least 10 applications. They show no individual application or value.

In [ ]:
nb.section("F-data")

D4 leaves out:
- **categorical features**, whose values have no order to rank: `payment_type`, `employment_status`, `housing_status`, `source`, `device_os`;
- **`device_fraud_count`**, which has the same value for every application;
- **`month`**, the time index used to split the data, not a property of the applicant.

(1) The applications as the dataset provides them, before any model. Descriptive.

(2) In every month, fraud is more frequent among applicants aged 50 or over than among younger applicants, and in point value its rate rises from one age group to the next. Age therefore carries information about the label, so a model that takes age as an input can learn to flag older applicants more often; the Alert Model and Classifier h both take it as an input. The Alert Model does flag a larger share of older legitimate applicants, and this part is the same under every policy. Among the features, `date_of_birth_distinct_emails_4w`, `phone_home_valid`, `credit_risk_score` and `current_address_months_count` are the most correlated with age, so they too carry information about age. None of this shows that age causes fraud, that the models are unfair by design, or that this is the only source of the age gaps in the results.

(3) It decides nothing.

**Reviewers.** The notebook generates the 50 reviewers with the dataset's own generator. This is not model fitting in the machine-learning sense: a reviewer does not learn from data. Each reviewer takes the true label of an alert and gets it wrong with a probability given by a fixed formula of the alert's features. The generator draws the formula's weights at random, then sets two offsets so that, on the alerts of months 3 to 6, the reviewer's FPR (legitimate applications wrongly rejected ÷ all legitimate) and FNR (frauds wrongly accepted ÷ all frauds) match targets drawn for it.

**Teams.** A team has ten reviewers and is run in five versions: five disjoint sets of ten, drawn at random from the 50 with a fixed seed. Within a version, the three teams are made of the same ten reviewers and differ only in how they decide a reviewed alert:
- **Team A, standard mixed team (the control):** members' own decision stands.
- **Team B, agrees with the model:** members follow the model in 4 of 5 reviewed cases, picked at random; otherwise their own decision stands.
- **Team C, biased:** members reject older applicants (aged 50 or over) whenever the model would.

The precise behaviour of each team is fixed by the teams' decision rules in the design. Capacity is not a property of a team: every team is run at every capacity.

**Models.** Two models are re-used from the dataset for efficiency reasons. Others are rebuilt.

_Re-used models_:
1. _Alert Model_ - a LightGBM model that assigns each application a fraud score (i.e. creates alerts)
2. _Classifier h_ - a LightGBM model that classifies alerts

_Trained models:_

_Routers_ - the models that predict how likely each reviewer is to decide an alert correctly: one model per reviewer, for each team version and each of five training shuffles, **250 in all**. The routers' hyperparameter tuning was run in advance, and by default the notebook refits the models from the settings it chose (the switch at the top).

**What is varied.** Capacity: 0 to 1 in steps of 0.05, plus 10/11 (22 values). Every combination of team, version, random order, capacity and cost ratio is run and reported.

**How to read the results.**

- **Capacity**: the share of the test month's alerts a team can review, from 0 (none) to 1 (all). The realistic range is 0.65 to 1, which includes 10/11, the share the benchmark gives its own reviewers. At 0.75 the project outline counts capacity as materially exceeded; there teams B and C are also compared with team A.
- **Cost ratio** λ: the cost of wrongly rejecting a legitimate applicant divided by the cost of missing a fraud. The main results use 0.057, the ratio Classifier h was trained for. Results for other values of λ are also shown (descriptive): 0.035, 0.0537 and 0.171.
- **Three measures, never combined; lower is better**: misclassification cost, frauds accepted (regulatory exposure), and the size of the age gap (fairness exposure). The first and the last are defined below.
- **Differences**: the policy named first minus the one it is compared with, on the same alerts. Below zero favours the policy named first.
- **Verdicts**: each difference has a range, from re-drawing the test month's alerts. ✅ **beats**: the whole range is below zero. ❌ **does worse**: the whole range is above zero. 🟡 **no difference shown**: the range includes zero. A star (✅*, ❌*, 🟡*) marks a deciding result.
- **Deciding and descriptive**: the deciding results are named in the design. Their ranges are widened so that, across all of them, the chance of a false finding stays near 5%. All other results are descriptive: they have ordinary 95% ranges and decide nothing. Where the two differ, the deciding verdict governs.
- **Versions**: a team beats or does worse only if all five versions do; otherwise the average of the five decides. Each version is run with five random orders of the alerts, and the results are averaged over them.
- **"By construction"** marks a figure that follows from the rules themselves: a check, never a finding.
- **A dash (—)** in a table marks an empty cell: the figure or note does not apply to that row.

> **Important: every FPR in this notebook is taken among the alerts, not over all applications.** The reason: the study assesses how effective human review is, and the alerts are its starting point. Applications the Alert Model did not flag, including the frauds it missed, are outside the study. Because the alerts are applications already flagged as suspicious, the FPRs here are much higher than a bank's rejection rate across all applicants.

**Age gap** = FPR<sub>50+</sub> − FPR<sub>&lt;50</sub> = (wrongly rejected 50+ ÷ legitimate 50+) − (wrongly rejected &lt;50 ÷ legitimate &lt;50)

**Size of the age gap** = | age gap |

**Misclassification cost** = FN + λ × FP, summed over the alerts (λ = cost ratio; 0.057 in the main results). The project outline calls it 'expected loss'.

<a id="3-hypothesis"></a>
## 3. Hypothesis

**What "deferral" means in this study.** The study replaces calibrated deferral with analyst-aware deferral (one of the aspirational extensions) as it better illustrates the objective.

The hypothesis has two parts. **First**, across realistic review capacity, analyst-aware deferral achieves lower risk than both full automation and naive review of every alert. **Second**, naive review can be worse than full automation when capacity is exceeded or when reviewers largely agree with the model.

**The three policies**, named by number in the tables:
- **policy 1, full automation**: Classifier h decides every alert;
- **policy 2, naive review**: every alert is meant for review; the team reviews as many as its capacity allows, and every alert beyond capacity is confirmed without review, so the application is rejected (i.e. zero regulatory exposure for alerts beyond capacity);
- **policy 3, analyst-aware deferral**: each alert is sent to either a reviewer or the model (Classifier h), decided based on their likeliness to decide correctly. Specifically, by comparing Classifier h confidence (the larger of its fraud and no-fraud probabilities) with router's score (how likely is that the analyst decides correctly), subject to team capacity and bias.

**The automated baseline.** Full automation reproduces the total cost the benchmark publishes for Classifier h on the test month (Alves et al., 2025, p. 24, Table 7).

**How the results are told.** The results are told policy by policy. Naive review (3.1) and analyst-aware deferral (3.2) are each compared with full automation, and analyst-aware deferral is also compared with naive review. The teams are the conditions each policy is tested under: team A, the control, at full capacity; team B, reviewers who agree with the model, and team C, reviewers biased against applicants aged 50 or over, both across the realistic range 0.65 to 1. The verdicts that decide are those of sections 3.1 and 3.2, judged in section 3.3. For each policy, team and measure, section 4.5 also gives the capacity range over which the policy beats full automation. Those ranges are descriptive: they decide nothing, and where one differs from a deciding verdict, the deciding verdict governs.

<a id="31-naive-review-against-full-automation"></a>
### 3.1 Naive review against full automation

The tables and graphs this notebook produces are computed from FiFAR (Alves et al., figshare, 2025, https://doi.org/10.6084/m9.figshare.28351172, CC BY 4.0, https://creativecommons.org/licenses/by/4.0/). They are derived from FiFAR by this project, not part of it, and FiFAR's creators have not reviewed or endorsed them.

> **Runtime estimate (Colab free): 5 minutes**
>
> The next cell regenerates the 50 reviewers, draws the teams, runs full automation and naive review, and re-draws the cases 50,000 times.

In [ ]:
nb.section("P2-F10")

(1) The grid shows every deciding result of naive review at cost ratio 0.057: one row per measure (for the age gap, its size), one column per team, capacity across, and the difference from full automation with its range; the colour gives the verdict. In the panels of teams B and C, a second marker shows the team against team A, both at capacity 0.75. These are the results that decide the second part of the hypothesis, and with section 3.2, the refutation conditions. They are judged only at the capacities the project outline judges: team A at full capacity, and teams B and C across the realistic range, 0.65 to 1; lower capacities are shown but not judged. The full sweep over every capacity is descriptive and decides nothing: see [section 4](#4-descriptive-results) and `tables/P2-T5_full_sweep.csv`.

(2) On misclassification cost, naive review beats full automation with team A at full capacity, with team B only at the highest realistic capacities, and with team C except at the lowest realistic capacities. On frauds accepted, it does worse with team A at full capacity and with team B only at full capacity, and shows no difference with team C. On the age gap, it beats with team A, with team B only at the lowest realistic capacity, and does worse with team C at every realistic capacity. Elsewhere in the realistic range, team B shows no difference. Within naive review at 0.75, team B does worse than team A on misclassification cost and beats it on frauds accepted; team C shows no difference from team A on misclassification cost, and its two other results follow from its rule.

(3) The second part is refuted on misclassification cost and not refuted on frauds accepted or the age gap (section 3.3).

<a id="32-analyst-aware-deferral-against-full-automation-and-naive-review"></a>
### 3.2 Analyst-aware deferral against full automation and naive review

> **Runtime estimate (Colab free): 7 minutes**
>
> The next cell builds the router's training files, refits its 250 models, routes the alerts under analyst-aware deferral and re-draws the cases 50,000 times.

In [ ]:
nb.section("P3-F10")

(1) The grid, drawn as for naive review, shows every deciding result of analyst-aware deferral at cost ratio 0.057; a second marker and colour show the three comparisons against naive review: team A at full capacity, and teams B and C at capacity 0.75. These are the results that decide the first part of the hypothesis: whether analyst-aware deferral beats full automation in every deciding result and naive review in all three comparisons. They are judged at the same capacities as naive review. The full sweep is descriptive: see [section 4](#4-descriptive-results) and `tables/P3-T5_full_sweep.csv`.

(2) On misclassification cost, analyst-aware deferral beats full automation with team A at full capacity and with teams B and C at every realistic capacity, and beats naive review with teams A and C. On frauds accepted, it does worse than full automation with teams A and B at full capacity, and shows no difference elsewhere. On the age gap, it beats full automation with team A, shows no difference with team B, and does worse with team C at every realistic capacity.

(3) Against full automation, the first part holds on misclassification cost and is refuted on frauds accepted and the age gap; against naive review, it is met on no measure (section 3.3).

**Age gap at capacity 0.75: FPR and FNR of each age group under each policy, teams B and C**

The graph below shows, for teams B and C at capacity 0.75, the FPR and FNR of each age group under each policy, because a smaller age gap can come from rejecting more applicants in both groups. FPR and FNR are among the alerts.

In [ ]:
nb.section("P3-F11")

(1) It shows whether a smaller age gap comes from treating the two groups more alike or from rejecting more applicants in both groups, and what each costs in missed frauds. Descriptive.

(2) With team B, the three policies keep the two age groups about equally far apart; analyst-aware deferral rejects fewer legitimate applicants than the other two in both age groups, and accepts more frauds in both. With team C, both review policies widen the gap mainly by rejecting far fewer legitimate applicants under 50, at the cost of more of their frauds accepted, while applicants aged 50 or over are rejected about as often as under full automation or more often. Under analyst-aware deferral, the wider gap comes almost entirely from the group under 50. In neither team does a policy narrow the gap by rejecting more applicants in both groups.

(3) The verdict on the age gap is the deciding one, in the grid above; this graph only explains it.

<a id="33-the-verdict-on-the-hypothesis-measure-by-measure"></a>
### 3.3 The verdict on the hypothesis, measure by measure

As the project outline requires, every combination is reported, including those unfavourable to the hypothesis.

The project outline gives two conditions under which the hypothesis is refuted. The study applies each to every measure separately:
- **First condition: "deferral fails to outperform automation at any capacity level".** Read strictly: if analyst-aware deferral fails to beat full automation in at least one of its deciding results against full automation (team A at full capacity; teams B and C at each realistic capacity), the hypothesis is refuted on that measure.
- **Second condition: "naive HITL never falls below automation".** If no team under naive review does worse than full automation in any of its deciding results against full automation, the hypothesis is refuted on that measure.

The second part of the hypothesis is judged with two further rules: naive review with capacity exceeded (team C below full capacity) and naive review by reviewers who agree with the model (team B). The first part is also judged against naive review: analyst-aware deferral must beat naive review in all three comparisons on a measure.

The rule-outcome files `tables/P2-T3c_rule_outcomes.csv` and `tables/P3-T3c_rule_outcomes.csv` apply the rules of the design to the deciding results of section 3, one measure at a time. Each row is one rule and one measure, with its outcome and the teams and capacities that produce it.

**Misclassification cost.** Analyst-aware deferral beats full automation in every one of its deciding results against it, so the first refutation condition is not met. Against naive review it beats with team A at full capacity and team C at 0.75, but shows no difference with team B at 0.75, so the rule that it beats naive review in all three comparisons is not met. Under naive review, no team does worse than full automation in any deciding result: team A at full capacity beats it, team B shows no difference up to 10/11 and beats at 0.95 and 1, and team C shows no difference at 0.65 and 0.70 and beats from 0.75. The second refutation condition is therefore met: on misclassification cost, the hypothesis is refuted. Neither the rule for exceeded capacity (team C) nor the rule for reviewers who agree with the model (team B) is supported. Team C's bias rule shows no difference in misclassification cost at 0.75 against the same reviewers without it.

**Frauds accepted.** Analyst-aware deferral does worse than full automation with teams A and B at full capacity and shows no difference elsewhere, so the first refutation condition is met: on frauds accepted, the hypothesis is refuted. Against naive review, it shows no difference with team A and does worse with teams B and C at 0.75. Under naive review, teams A and B at full capacity do worse than full automation, where the reviewers themselves do worse than the classifier on the alerts they review, so the second refutation condition is not met. The rule for reviewers who agree with the model is supported on this measure (team B at full capacity), but not through the agreement. The cause is that the reviewers themselves do worse than the classifier on the alerts they review. Agreeing with the model moves team B towards the classifier: at full capacity, team B's excess of frauds accepted over full automation is about a fifth of team A's (0.21; long notebook, A.9). The rule for exceeded capacity is not supported (team C shows no difference at every realistic capacity).

**Size of the age gap.** Analyst-aware deferral beats full automation with team A at full capacity, shows no difference with team B and does worse with team C at every realistic capacity, so the first refutation condition is met: on the age gap, the hypothesis is refuted. Against naive review, it shows no difference with teams A and B and does worse with team C at 0.75. Under naive review, team C does worse than full automation at every realistic capacity, so the second refutation condition is not met, and the rule for exceeded capacity is supported; team C's age gap comes from its rule (section 4.7). The rule for reviewers who agree with the model is not supported: team B beats full automation at 0.65 and shows no difference elsewhere.

None of this shows what real reviewers, a better router or a cost of review would give.

<a id="4-descriptive-results"></a>
## 4. Descriptive results

<a id="41-where-the-difference-against-naive-review-comes-from"></a>
### 4.1 Where the difference against naive review comes from

**Analyst-aware deferral minus naive review, split into the alerts left to the classifier and the alerts reviewed, on both measures, with ranges and verdicts**

The graph gives, for team A at full capacity and teams B and C at capacity 0.75, the part from the alerts analyst-aware deferral leaves to the classifier and the part from the alerts it sends to a reviewer, on misclassification cost at cost ratio 0.057 and on frauds accepted. Each marker is the average of the five versions, with its 95% range, coloured by its verdict and labelled with its value. These results are descriptive and decide nothing. They are read from the run's full results by their key columns, with nothing recomputed; the cost ratio does not apply to frauds accepted.

In [ ]:
nb.section("rows-split")

(1) It shows whether analyst-aware deferral's difference from naive review comes from the alerts it reviews or from the alerts it leaves to the classifier, which naive review would confirm beyond capacity. Only the first bears on routing to the right reviewer. Descriptive.

(2) In the three deciding comparisons, the advantage on misclassification cost lies in the alerts reviewed.

(3) Team B's extra frauds accepted lie in the alerts left to the classifier, so they reflect how the alerts not reviewed are treated, not the routing; team C's lie in the alerts reviewed. The verdict of the whole difference is the deciding verdict of section 3.2; the verdicts of the parts are descriptive.

The split does not separate the router's choice of alerts and reviewers from the way the alerts it does not review are treated. Its results are descriptive and decide nothing; the graph above gives both parts on both measures, with the 95% ranges and verdicts.

<a id="42-how-analyst-aware-deferral-routed-the-alerts-and-how-well-the-router-ranks"></a>
### 4.2 How analyst-aware deferral routed the alerts, and how well the router ranks

The project outline proposes that a case should reach a human only if (1) the reviewer is more likely than the model to be correct; (2) the volume routed fits the actual review capacity; (3) the misclassification cost avoided (the project outline's "expected loss avoided") exceeds the cost of review; (4) the reviewer has the information and time to disagree with the model. In this study:
- **Criterion 1** is built into analyst-aware deferral as the router sees it: a reviewer takes an alert only if the router's estimate of the reviewer being right is strictly above the classifier's confidence. Whether it holds in fact is measured (section 4.9).
- **Criterion 2** is built in: never more alerts are reviewed than capacity allows.
- **Criterion 3** is measured, not enforced: the cost avoided per review is shown as the cost of one review at which review breaks even (section 4.9). No cost of review is modelled.
- **Criterion 4** cannot be observed directly in a simulation without time pressure; it shows only through how often reviewers overturn the alert (section 4.9).

**Analyst-aware deferral: how the alerts are allocated against capacity, and how the reviewed alerts are shared among the ten reviewers**

The first graph shows, at capacities 1, 10/11, 0.75 and 0.65, the share of the 4,457 alerts routed to a reviewer, routed to the classifier, and decided by the classifier because the pool of review places is full, counted by the branch of the routing trees and averaged over the five versions and random orders. Teams A and B share one routing; team C has its own. It shows how the share decided by the classifier because the pool is full grows as capacity falls (criterion 2). The second graph shows, for teams A and B at capacities 1 and 0.75, each reviewer's share of the reviewed alerts, from the busiest to the least busy; a flat line would be an even split. The counts per team, version, random order and capacity are in `tables/P3-T1a_routing.csv`.

In [ ]:
nb.section("P3-T1")

**How well the router tells apart the alerts each reviewer gets right from those the reviewer gets wrong**

Analyst-aware deferral relies on the framework's **OvA Classifiers**, one model per reviewer, here called the router, to estimate how likely each reviewer is to decide an alert correctly. The router's models are trained on the decisions of this study's reviewers by the framework's own procedure, including its settings search; by default this notebook refits them from the settings that search chose (section 2). How well these estimates separate the alerts a reviewer decides correctly from those the reviewer gets wrong is measured as an area under the ROC curve: 0.5 is chance and 1 is perfect. The graph shows, for each team version, the average, lowest and highest of this figure over the version's 50 models (its ten reviewers, five training shuffles each), on the test month beside the previous month. The previous month is the one on which the router's models were validated and their settings chosen, so its figures are not measured on unseen data; only the test month's are.

(Each figure is the area under the ROC curve with legitimate applicants weighted 0.057 and frauds 1. The figure for each single model is in the accompanying table file.)

In [ ]:
nb.section("P3-T1r")

Averaged over each version's models, the router ranks only a little better than chance, and about equally in the two months; 6 of the 250 single models are at or below chance on the test month, and 1 on the previous month. Because the previous month is the router's validation month, the agreement between the two months says little about how stable the ranking is. The router ranks the reviewers' correctness only weakly. Analyst-aware deferral is therefore tested as the framework implements it, not as the best possible router would do.

<a id="43-the-reviewers-against-classifier-h"></a>
### 4.3 The reviewers against Classifier h

The graph shows how good each of the 50 reviewers is compared with Classifier h, on the alerts of the test month.

In [ ]:
nb.section("P2-T1")

(1) Criterion 1 of the project outline asks whether the reviewer is more likely than the model to be correct. The graph shows how far that holds for each reviewer on its own, before any team rule or routing. A reviewer above the dashed line beats Classifier h on misclassification cost as h is set. A reviewer above h's curve does better than h could at any cut-off. Descriptive.

(2) Every reviewer lies above Classifier h's curve: on its own, each separates frauds from legitimate applicants better than h could at any cut-off. All but two lie above the dashed line, so they have a lower misclassification cost than h as it is set; the other two lie just below it. The reviewers differ widely in how many legitimate applicants they reject.

(3) It decides nothing; section 4.9 shows criterion 1 as it turns out on the alerts actually reviewed.

<a id="44-misclassification-cost-against-capacity"></a>
### 4.4 Misclassification cost against capacity

Full automation, the baseline, has at cost ratio 0.057 a misclassification cost of 204.05, accepts 40 of the 711 frauds among the alerts (FNR 5.6%), and has an age gap of 0.0696 (FPR 0.8166 at 50 or over, 0.7470 under 50, among the alerts); its figures at every cost ratio, with ranges, are in `tables/P1-T2_baseline_by_cost_ratio.csv`.

**Analyst-aware deferral minus full automation, and minus naive review, misclassification cost, against capacity**

Each panel is one team. Blue is the difference from full automation, red the difference from naive review; for each, the thin lines are the five versions and the thick line their average with its 95% band. The realistic range, 0.65 to 1, is shaded. Below zero, analyst-aware deferral has the lower misclassification cost.

In [ ]:
nb.section("P3-F1")

(1) It shows over the whole capacity grid where analyst-aware deferral's misclassification cost lies against full automation and against naive review, including capacities outside the realistic range. Descriptive.

(2) Analyst-aware deferral beats full automation for every team from the lowest capacities above zero, and beats naive review at every realistic capacity on the ordinary ranges.

(3) It supports the first part on misclassification cost against full automation; against naive review, the deciding result for team B shows no difference (section 3.2).

<a id="45-the-capacity-range-over-which-each-policy-beats"></a>
### 4.5 The capacity range over which each policy beats

**Naive review: the capacity range over which it beats full automation, per team and measure, and the verdict at each capacity**

The graph gives, for each team and measure, the descriptive verdict at every capacity of the grid, with a tick where the average difference crosses zero. The crossing points and the verdicts of each version are in `tables/P2-T4_capacity_boundaries.csv`. Where a cell at a realistic capacity is also a deciding result, the deciding verdict in section 3 governs.

In [ ]:
nb.section("P2-T4")

(1) For each team and measure, the capacities at which the policy beats full automation, does worse, or shows no difference. Descriptive: these ranges decide nothing. Where a cell at a realistic capacity is also a deciding result, the deciding verdict in section 3 governs.

(2) On misclassification cost, naive review never does worse than full automation, and beats it from some capacity upwards with every team: earliest with team A, latest with team B. On frauds accepted, it beats at low capacities, because the alerts beyond capacity are confirmed, and does worse near full capacity with every team. On the size of the age gap, it beats with teams A and B at almost every capacity, and does worse with team C over most of the grid, by team C's rule.

(3) The judging capacities of the project outline (team A at full capacity, teams B and C from 0.65 to 1) are fixed inputs to the design. These ranges show where each policy's advantage holds inside and outside them; they do not test the judging capacities.

**Analyst-aware deferral: the capacity range over which it beats full automation and naive review, per team and measure, and the verdict at each capacity**

The two graphs give, for each team and measure, the descriptive verdict at every capacity of the grid, with a tick where the average difference crosses zero. The crossing points and the verdicts of each version are in `tables/P3-T4_capacity_boundaries.csv`. Where a cell at a realistic capacity is also a deciding result, the deciding verdict in section 3 governs.

In [ ]:
nb.section("P3-T4")

(1) For each team and measure, the capacities at which the policy beats full automation (and naive review), does worse, or shows no difference. Descriptive: these ranges decide nothing. Where a cell at a realistic capacity is also a deciding result, the deciding verdict in section 3 governs.

(2) On misclassification cost, analyst-aware deferral beats full automation with every team at almost every capacity, and beats naive review with every team from a low capacity upwards, earliest with team C and latest with team B. On frauds accepted, it does worse than full automation with every team over most of the grid, and worse than naive review at every capacity up to 0.85 with team A and up to 10/11 with teams B and C. On the size of the age gap, against full automation it beats with team A except at the lowest capacities, shows no difference with team B, and does worse with team C at every capacity above 0. Against naive review, its gap is the larger with team C at every capacity (at capacity 0 by construction), with team B at all but the highest, and with team A only below the middle of the grid.

(3) The judging capacities of the project outline (team A at full capacity, teams B and C from 0.65 to 1) are fixed inputs to the design. These ranges show where each policy's advantage holds inside and outside them; they do not test the judging capacities.

<a id="46-who-gets-a-reviewer-access-by-age-group"></a>
### 4.6 Who gets a reviewer: access by age group

**Team C: share of alerts sent to a reviewer against capacity, by age group, analyst-aware deferral and naive review**

For teams A and B, both age groups reach a reviewer in proportion to capacity under both policies; their lines lie on naive review's diagonal, so they are not drawn. The graph shows team C: solid lines for analyst-aware deferral, dashed for naive review; red for applicants aged 50 or over, blue for those under 50.

In [ ]:
nb.section("P3-F9")

(1) It shows whether older and younger applicants get the same access to a reviewer under each policy, which bears on fairness exposure beyond the outcome.

(2) Under naive review, both age groups reach a reviewer in proportion to capacity. Under analyst-aware deferral with team C, from a capacity of 0.75 upwards, applicants aged 50 or over reach a reviewer only about one time in six (16%), while applicants under 50 almost always do (99%); below 0.75 both shares are lower, and the older group's never exceeds 16%. None of the older applicants whose alert the classifier would confirm ever reaches a reviewer: team C's rule would reject them whatever the reviewer decided, so the routing sends them straight to the classifier.

(3) With a biased team, analyst-aware deferral quietly denies most older applicants a human review, although the outcomes may look equal on average. This decides nothing.

In the table below, within each age group, the policies keep the same order as among the alerts, because each group's denominator is fixed.

In [ ]:
nb.section("P3-T10")

<a id="47-team-cs-bias-against-capacity"></a>
### 4.7 Team C's bias against capacity

**Naive review: FPRs by age group for teams C and A against capacity, and how team C's bias changes between capacity 0.75 and full capacity**

The project outline's question is whether a bias compounds or cancels when capacity is exceeded. The measure is the difference between teams C and A at capacity 0.75, minus the same difference at full capacity, on three figures: the signed age gap, the number of wrongly rejected applicants aged 50 or over, and misclassification cost. Above zero, the bias compounds with overload; below zero, it cancels in part; otherwise it is additive. As for every difference, a verdict of "beats" on this change means only that its whole range is below zero, and "does worse" that it is above zero. For naive review, the graph also shows both teams' FPRs by age group against capacity, and team C's gap over team A against the line expected if the bias simply scaled with capacity.

In [ ]:
nb.section("P2-F5")

Team C's excess over team A on the age gap and on the number of wrongly rejected applicants aged 50 or over shrinks rather than grows between full capacity and 0.75, a check that follows from the rules; the one finding is on misclassification cost, where the verdict is "beats" and the reading "cancels (in part)" (three of the five versions; two read "additive"), and neither says anything about real reviewers.

<a id="48-other-cost-ratios"></a>
### 4.8 Other cost ratios

**Naive review: misclassification-cost verdict against full automation at each cost ratio and capacity**

Each grid shows, for each team, the descriptive misclassification-cost verdict against full automation at every capacity (across) and cost ratio (down): green beats, red does worse. Away from 0.057, neither full automation nor the reviewers (nor, for analyst-aware deferral, the router) is re-tuned for the cost ratio, so every "beats" there holds only "against an automation baseline not re-optimised for this cost ratio".

In [ ]:
nb.section("P2-F7")

(1) The cost ratio is a parameter of the study. The graph shows whether naive review's verdict on misclassification cost holds away from the main ratio, against a baseline not re-tuned for it. Descriptive.

(2) Where naive review beats full automation with a team at the main cost ratio, it also beats at the highest ratio and, with one exception for team A, at the ratio just below the main one; at the lowest ratio most of these turn to no difference, and at the highest every team does worse at low capacities.

(3) No naive-review result is shown to beat full automation robustly across cost ratios, because full automation is not re-tuned for each ratio. The run's tables label a team and capacity "conditional" if it beats full automation at one or more of the other cost ratios and does worse at none, and "not robust" if it beats at one or more and does worse at one or more. Even where a team beats full automation at all four ratios (team C at the higher capacities), the label is "conditional"; team C is labelled "not robust" at capacities 0.15 to 0.25.

**Analyst-aware deferral: misclassification-cost verdict against full automation at each cost ratio and capacity**

In [ ]:
nb.section("P3-F6")

(1) As P2-F7, for analyst-aware deferral. Descriptive.

(2) Analyst-aware deferral beats full automation with teams A and C at every capacity above 0, at all four cost ratios. With team B, it beats only from a middle capacity upwards; below that, it shows no difference at the lowest cost ratio, and at the lowest capacity above zero it beats only at the highest ratio.

(3) Every team is labelled "conditional" at every capacity above 0: it beats full automation at one or more of the other cost ratios and does worse at none. No result is labelled "not robust" (beats at one or more, does worse at one or more), and none is shown against a baseline re-tuned for the cost ratio.

<a id="49-governance-effectiveness-monitoring"></a>
### 4.9 Governance effectiveness monitoring

The project outline proposes four indicators a bank could compute from its own review logs: reviewer override rate, capacity utilisation, cost avoided per review, and subgroup disparity. The graphs show them for each team against capacity, averaged over the five versions, with 95% bands where a range exists. They are not defined at capacity 0, where nothing is reviewed. Two are adapted. Under naive review every review place is always used, so capacity utilisation is shown as the number of alerts per review place (1 ÷ capacity, the same for every team); under analyst-aware deferral, as the share of review places used. Cost avoided per review, against full automation, is the cost of one review at which review breaks even, in units of a missed fraud (criterion 3). For analyst-aware deferral, two more panels show criterion 1 as it turns out: on reviewed alerts, the share where the reviewer is right and full automation wrong, and the reverse.

In [ ]:
nb.section("P2-F6")

(1) These are the indicators the project outline proposes for detecting inappropriate oversight from review logs. No rule was fixed for them; they decide nothing.

(2) Team A overturns the most reviewed alerts, team B the fewest and team C in between, almost independently of capacity; team B's cost avoided per review is about a fifth of team A's; team C's age gap on reviewed alerts is much the largest.

(3) The indicators separate the three teams clearly, but no rule turns them into a verdict.

**Analyst-aware deferral: monitoring indicators against capacity, per team, with criterion 1 as it turns out**

In [ ]:
nb.section("P3-F5")

(1) As P2-F6, and it shows criterion 1 as it turns out: how often the reviewer is right where the classifier is wrong, and the reverse. Descriptive.

(2) Teams A and B use every review place except near full capacity, where they use slightly fewer; team C uses fewer from a lower capacity, because fewer of its alerts are routed to a reviewer. On reviewed alerts, the reviewer is right where the classifier is wrong far more often than wrong where the classifier is right.

(3) Criterion 1 holds on the reviewed alerts as it turns out; the indicators decide nothing.

No rule was fixed for these indicators, so they carry no verdict and support no conclusion about detecting inappropriate oversight before it fails. They show what such indicators look like in this simulation. Criterion 4 shows only through the override rate. Two readings follow from the rules, and both hold in this run: under naive review, the load is the same for every team at a given capacity, and team B's cost avoided per review is about a fifth of team A's.

In [ ]:
# Writes the table and graph files of the items only the long notebook shows; shows nothing.
nb.write_files("P1-T2", "P1-F1", "P2-F1", "P2-F4", "P3-F4", "P2-F2", "P3-F2", "P2-F3", "P3-F3", "P3-T7", "P3-T9", "rows-0035", "P2-T5", "P3-T5", "P2-F8", "P3-F7")

Every team, capacity, version and cost ratio, with all measures: `tables/P2-T5_full_sweep.csv`, `tables/P3-T5_full_sweep.csv`.

<a id="5-limitations-and-risks"></a>
## 5. Limitations and risks

<a id="51-limitations"></a>
### 5.1 Limitations

- **The reviewers are simulated.** Each makes one fixed decision per alert; there is no fatigue, time pressure or learning, and exceeding capacity changes only what happens to the alerts beyond it. The project outline's criterion 4 shows only through the override rate.
- **No cost of review is modelled.** Cost avoided per review is shown as the break-even cost instead.
- **One alert model, one test month:** 4,457 alerts, 711 of them frauds. Frauds the Alert Model did not flag are outside the study.
- **The cost ratio is implied, not observed.** The data suggest a ratio of roughly 0.03 to 0.045; of that range only 0.035 is tested. Away from 0.057, neither full automation, the reviewers nor the router is re-tuned, and how much better a re-tuned automation would do is not measured.
- **The reviewers are regenerated**, without any weight on age and fitted on four months; they are not the dataset's original reviewers.
- **Team B's 4 in 5** is the author's judgement, not observed behaviour; other values would give other results for team B. **Team C's bias is a rule**, not a model of how bias arises in people.
- **The router is weak.** It ranks the reviewers' correctness only weakly, and some of its models are at or below chance. A router that fits the reviewers less well puts analyst-aware deferral at a disadvantage.
- **The routing's own effect is not separated** from how the alerts not reviewed are treated (section 4.1).
- **Capacity is modelled as one shared pool** of review places, with the classifier unlimited, and the router sends most of the work to a few reviewers. With teams A and B at full capacity, the two busiest of the ten reviewers take between about two in five and four in five of the reviewed alerts, depending on the version and the random order. In more than half of the runs at least one reviewer takes none (section 4.2). Which reviewers are favoured changes with the router's training shuffle. No real team could absorb such a split, and fatigue is not modelled, so analyst-aware deferral is shown under conditions more favourable than a real team would offer.
- **The alerts are taken in a random order**, not by score, so that the policies are compared like for like; under analyst-aware deferral the order affects which alerts are reviewed, and its effect is part of the spread over the random orders.
- **The ranges cover which cases fell in the test month**, not the policy's own randomness, which is shown as the range over random orders. The rule that a team beats or does worse when all five versions agree, or else by their average, makes the 5% chance of a false finding approximate. The deciding results of the two policies are adjusted separately, so across both together that chance is above 5%.
- **Capacity steps by 0.05**, so every crossing is located only to within that step.
- **The implementation was checked during the study.** The checks are not repeated in this notebook.
- The project outline's own limitation on the author's dual role as project owner and stakeholder stands.

<a id="52-risks"></a>
### 5.2 Risks

The risk named in the project outline, that the pre-trained models might not run well on the available Apple Silicon machine, did not stop any part of the study: the study's own computation ran on the project's machine. The outputs shown in this notebook were computed on free Google Colab, by the run that produced this copy. Run time and memory are in the technical section.

# Technical section

This section is not needed to read the results. It holds the state of the switch in this run, the input checksums and their outcome, the map from each item above to the study design's identifiers and to the files the run writes, the label texts, the full deciding tables with a graph of each result and its range, and the caveat on the boundary summaries. It ends with the run time and peak memory of every step of this run, and with the cell that collects the run's outputs in one archive. The study design's identifiers appear here.

<a id="the-switch-in-this-run"></a>
## The switch in this run

In [ ]:
nb.switch_state()

<a id="input-checksums"></a>
## Input checksums

In [ ]:
nb.input_checksums()

<a id="map-from-each-item-to-its-design-identifier"></a>
## Map from each item to its design identifier

Each row names an item of the notebook, the identifier of the table or graph in the study's design that it shows, and the files this run writes for it, in its folders `tables/` and `figures/`. The two grids P2-F10 and P3-F10 draw the deciding results of P2-T3 and P3-T3; rows-split and rows-0035 are rows of the full sweep tables P3-T5 and P2-T5, read by their key columns. The files in `tables/` keep the study's internal column names: columns named with "expected_loss" hold the misclassification cost, and columns named with "false_rejection_rate" hold the FPR (likewise "false_rejections" holds the FP count and "miss_rate" the FNR).

| Order | Section | Item | Design identifier | Files the run writes |
|---|---|---|---|---|
| 0a | 2 | Which months the study uses, and for what | F-months (new) | `figures/F-months_months_used.png` |
| 0b | 2 | What the applications show: label, age and features (panels D1 to D5) | F-data (new) | `figures/F-data_applications.png` |
| 1 | 3.1 | Naive review: the results that decide, at cost ratio 0.057, in one grid | P2-F10 (from P2-T3) | `figures/P2-F10_deciding_results.png`; `tables/P2-T3_deciding_results.csv`, `tables/P2-T3b_realistic_range_verdicts.csv`, `tables/P2-T3c_rule_outcomes.csv` |
| 2 | 3.2 | Analyst-aware deferral: the results that decide, at cost ratio 0.057, in one grid; FPR and FNR of each age group at 0.75 (new) | P3-F10 (from P3-T3); P3-F11 (new) | `figures/P3-F10_deciding_results.png`; `tables/P3-T3_deciding_results.csv`, `tables/P3-T3b_realistic_range_verdicts.csv`, `tables/P3-T3c_rule_outcomes.csv`; `figures/P3-F11_fpr_fnr_by_age.png` |
| 3 | A.2 (long notebook) | Analyst-aware deferral minus naive review, misclassification cost, split into the alerts left to the classifier and the alerts reviewed | P3-F4 | `figures/P3-F4_split.png` |
| 4 | — | Teams B and C at capacity 0.75: the difference on the alerts naive review confirms without review (file only; no longer shown) | P3-T8 | `tables/P3-T8_point_figures.csv` |
| 5 | 4.1 | Analyst-aware deferral minus naive review, split into the alerts left to the classifier and the alerts reviewed, on both measures, with ranges and verdicts | rows of P3-T5, as a range graph | `tables/rows-split.csv`, `figures/rows-split_parts_range.png` |
| 6 | 4.2 | Analyst-aware deferral: how many alerts went to a reviewer, to which reviewer, and to the classifier | P3-T1 (routing), as two graphs | `tables/P3-T1a_routing.csv`, `figures/P3-T1_allocation.png`, `figures/P3-T1_reviewer_shares.png` |
| 7 | 4.2 | How well the router tells apart the alerts each reviewer gets right from those the reviewer gets wrong | P3-T1 (ranking quality) | `tables/P3-T1b_ranking_quality_per_model_month6.csv`, `tables/P3-T1b_ranking_quality_per_model_month7.csv`, `tables/P3-T1c_ranking_quality_by_version.csv`, `figures/P3-T1r_ranking_quality.png` |
| 8 | 4.3 | The 50 reviewers against Classifier h on the test month, in ROC space (the P2-T1 tables as files only) | P2-T1 | `tables/P2-T1a_members_slot_order.csv`, `tables/P2-T1c_team_version_rates.csv`, `tables/P2-T1d_twins.csv`, `tables/P2-T1e_month7_items.csv`, `figures/P2-T1_reviewers_roc.png` |
| 9 | A.1 (long notebook) | Full automation and "confirm every alert": the three measures at each cost ratio | P1-T2 | `tables/P1-T2_baseline_by_cost_ratio.csv` |
| 10 | A.1 (long notebook) | Full automation and "confirm every alert": the three measures against the cost ratio | P1-F1 | `figures/P1-F1_measures_vs_cost_ratio.png` |
| 11 | A.2 (long notebook) | Naive review minus full automation, misclassification cost, against capacity | P2-F1 | `figures/P2-F1_expected_loss.png` |
| 12 | 4.4 | Analyst-aware deferral minus full automation, and minus naive review, misclassification cost, against capacity | P3-F1 | `figures/P3-F1_expected_loss.png` |
| 13 | A.2 (long notebook) | Naive review minus full automation, misclassification cost, split into the reviewed alerts and the alerts beyond capacity | P2-F4 | `figures/P2-F4_split.png` |
| 14 | A.3 (long notebook) | Naive review minus full automation, frauds accepted, against capacity | P2-F2 | `figures/P2-F2_accepted_frauds.png` |
| 15 | A.3 (long notebook) | Analyst-aware deferral minus full automation, and minus naive review, frauds accepted, against capacity | P3-F2 | `figures/P3-F2_accepted_frauds.png` |
| 16 | A.4 (long notebook) | Naive review minus full automation, size of the age gap, against capacity, with the overall FPR | P2-F3 | `figures/P2-F3_absolute_age_gap.png` |
| 17 | A.4 (long notebook) | Analyst-aware deferral minus full automation, and minus naive review, size of the age gap, against capacity, with the overall FPR | P3-F3 | `figures/P3-F3_absolute_age_gap.png` |
| 17a | 4.6 | FPRs on every application of the test month, descriptive | P3-T10 (new) | — (shown only) |
| 18 | 4.5; table A.5 (long notebook) | Naive review: the capacity range over which it beats full automation, per team and measure, and the verdict at each capacity | P2-T4, with its colour grid | `tables/P2-T4_capacity_boundaries.csv`, `figures/P2-T4_verdict_grid.png` |
| 19 | 4.5; table A.5 (long notebook) | Analyst-aware deferral against full automation: the descriptive verdict at each capacity; analyst-aware deferral against naive review: the descriptive verdict at each capacity | P3-T4, with its colour grid | `tables/P3-T4_capacity_boundaries.csv`, `figures/P3-T4_verdict_grid_against_full_automation.png`, `figures/P3-T4_verdict_grid_against_naive_review.png` |
| 20 | 4.7; table A.6 (long notebook) | Naive review: FPRs by age group for teams C and A against capacity, and how team C's bias changes between capacity 0.75 and full capacity | P2-F5 | `figures/P2-F5_bias_x_capacity.png`, `tables/P2-F5_bias_x_capacity_I.csv` |
| 21 | A.6 (long notebook) | Analyst-aware deferral: how the difference between teams C and A changes between capacity 0.75 and full capacity | P3-T7 | `tables/P3-T7_bias_x_capacity.csv` |
| 22 | A.7 (long notebook) | Share of alerts sent to a reviewer, by age group, analyst-aware deferral beside naive review | P3-T9 | `tables/P3-T9_access_by_age.csv` |
| 23 | 4.6 | Team C: share of alerts sent to a reviewer against capacity, by age group, analyst-aware deferral and naive review | P3-F9 | `figures/P3-F9_access_by_age.png`, `tables/P3-F9_lines.csv` |
| 24 | 4.8; tables A.8 (long notebook) | Naive review: misclassification-cost verdict against full automation at each cost ratio and capacity; how each team's result holds across cost ratios | P2-F7 | `figures/P2-F7_cost_ratio_grid.png`, `tables/P2-F7_cost_ratio_labels.csv` |
| 25 | 4.8; table A.8 (long notebook) | Analyst-aware deferral: misclassification-cost verdict against full automation at each cost ratio and capacity; how each team's result holds across cost ratios | P3-F6 | `figures/P3-F6_cost_ratio_grid.png`, `tables/P3-F6_cost_ratio_labels.csv` |
| 26 | A.8 (long notebook) | Naive review and analyst-aware deferral against "confirm every alert", misclassification cost at cost ratio 0.035 | rows of P2-T5 and P3-T5, as a range graph | `tables/rows-0035.csv`, `figures/rows-0035_range.png` |
| 27 | 4.9 | Naive review: monitoring indicators against capacity, per team | P2-F6 | `figures/P2-F6_monitoring.png` |
| 28 | 4.9 | Analyst-aware deferral: monitoring indicators against capacity, per team, with criterion 1 as it turns out | P3-F5 | `figures/P3-F5_monitoring.png` |
| 29 | A.10 (long notebook) | Naive review: every team, version, capacity and cost ratio, all measures | P2-T5 | `tables/P2-T5_full_sweep.csv` |
| 30 | A.10 (long notebook) | Analyst-aware deferral: every team, version, capacity and cost ratio, all measures | P3-T5 | `tables/P3-T5_full_sweep.csv` |
| 31 | A.11 (long notebook) | Naive review: observed "beats" and "does worse" against the number expected by chance | P2-F8 | `figures/P2-F8_chance_count.png` |
| 32 | A.11 (long notebook) | Analyst-aware deferral: observed "beats" and "does worse" against the number expected by chance | P3-F7 | `figures/P3-F7_chance_count.png` |
| 33 | technical | Naive review: the results that decide, in full, with their widened ranges | P2-T3, P2-F9 | the three P2-T3 files of row 1; `figures/P2-F9_forest.png` |
| 34 | technical | Analyst-aware deferral: the results that decide, in full, with their widened ranges | P3-T3, P3-F8 | the three P3-T3 files of row 2; `figures/P3-F8_forest.png` |
| 35 | technical | Run time and memory of this run | T-T1 (this run's own records) | — (no file of its own; the table is formed from the run's `logs/steps.jsonl`, which the archive holds) |

<a id="label-texts"></a>
## Label texts

The texts of the labels the tables carry, as the study's design sets them. Sentences that refer to the study's own checks and run records, and references to its internal decision numbers, are left out; nothing else is changed.

In [ ]:
nb.section("T-labels")

<a id="the-results-that-decide-in-full"></a>
## The results that decide, in full

These are the deciding results of sections 3.1 and 3.2 as tables: for naive review the study design's table P2-T3, for analyst-aware deferral P3-T3, each from the files named in section 3. Every deciding result is listed with its widened range and its verdict. Below each table, a graph draws each result with its widened range: a square for the average of the five versions and dots for the five versions. The two results of naive review that follow from team C's rule are shown apart and are not findings. The verdicts are the same as in the grids of sections 3.1 and 3.2.

In [ ]:
nb.section("T-P2-T3")

In [ ]:
nb.section("T-P3-T3")

<a id="boundary-summaries-caveat"></a>
## Boundary summaries: caveat

The two table files of section 4.5 hold a column headed "summary (can be misread; see the caveat in the technical section)" (`tables/P2-T4_capacity_boundaries.csv` and `tables/P3-T4_capacity_boundaries.csv`, column `interval summary`). The rule outcomes of naive review hold summaries of the same form in their table file, which is not displayed (`tables/P2-T3c_rule_outcomes.csv`, column `boundary_summary`). This run's verdict step forms these summaries once, and the table files hold them unchanged. Where a summary reads "does worse up to capacity X; never beats", with no "between:" list, it states only the highest capacity at which the comparison does worse, and that it never beats. It does not mean "does worse at every capacity up to X". The per-capacity verdicts govern: for the rule outcomes, those of `tables/P2-T3b_realistic_range_verdicts.csv`; for section 4.5, the colour grids and the capacity columns of the two table files (average of the five versions; the rows of each version are in the table files). No verdict, rule outcome or number changes.

In this run:
- `tables/P2-T3c_rule_outcomes.csv`: one row, the rule for reviewers who agree with the model on frauds accepted ("does worse up to capacity 1.00; never beats"): team B shows no difference at 0.65 to 0.95 and does worse only at 1.
- `tables/P2-T4_capacity_boundaries.csv`: no row has this form.
- `tables/P3-T4_capacity_boundaries.csv`: 27 rows have this form, all with X = 1.00. The 8 rows against naive review (team A version 2 on frauds accepted; team C version 2 on frauds accepted; team C versions 1 to 5 and the mean on the age gap) read "does worse" at every capacity. In the 19 rows against full automation, the only other verdict is "no difference shown": at capacity 0, where analyst-aware deferral is full automation, in every row; and also, on frauds accepted, for team A version 1 at 0.05, version 2 at 0.10, version 4 at 0.05 to 0.20 and version 5 at 0.05 to 0.15; for team B version 1 at 0.05, 0.10, 0.20 and 0.55, version 2 and the mean at 0.10; and for team C version 2 and the mean at 0.05 to 0.15. The other rows of this form (frauds accepted: team A version 3 and the mean, team B version 3, team C version 3; the age gap: team C versions 1 to 5 and the mean) differ from "does worse" at capacity 0 only.

<a id="run-time-and-memory-of-this-run"></a>
## Run time and memory of this run

Each step's wall-clock time and peak memory, in bytes, computed from this run's own records while it ran. The set-up steps (environment, framework and FiFAR) run in the notebook's own process, whose memory is not measured step by step, so their memory cell shows a dash.

In [ ]:
nb.run_time_table()

<a id="this-runs-outputs-in-one-archive"></a>
## This run's outputs in one archive

The last cell collects a description of this run's outputs in one archive, `savehitl_returned_outputs.zip`, so that they can be kept after the session ends; on Colab it also offers the archive for download. Among other records of the run, it holds the table files, the per-column summaries of the run's case-level files (for columns of numbers: count, sum, sum of absolute values, minimum and maximum; for other columns: count, number of distinct values and a checksum of the values), the checksums of the router's 250 refitted models, and the run's time and memory records. It holds no case row of FiFAR. Nothing else in the notebook depends on this cell.

In [ ]:
nb.hand_back()